# 3. Docking Cyclodextrins With AutoDock Vina (V02)

This notebook follows the **AutoDock Vina basic docking tutorial workflow** more directly, using notebook cells and command-line style Vina execution instead of importing helper functions from `docking_functions.py`.

Tutorial reference:
- [AutoDock Vina basic docking tutorial](https://autodock-vina.readthedocs.io/en/latest/docking_basic.html#)

What this notebook does:
- loads the already prepared receptor and ligand files from `Docking_Vina/prepared`
- defines a Vina box and writes a tutorial-style config TXT file for each host
- runs `vina --receptor ... --ligand ... --config ... --out ...`
- parses the docking results from the output `PDBQT` files
- visualizes the pre-docking setup and the docked poses

Important scope note:
- this V02 notebook focuses on the **standard Vina basic-docking workflow**
- the published B/C-form classification workflow from the other project version is **not used here**, because this lab folder only contains the standalone prepared receptor/ligand inputs


## 1. Set Up The Computing Environment


In [ ]:
# Installing necessary packages
%pip -q install vina py3Dmol pandas numpy matplotlib


In [ ]:
# Print versions of vina py3Dmol pandas numpy matplotlib
import sys
import numpy as np
import pandas as pd
import matplotlib
import py3Dmol
import vina

print('Python:', sys.version.split()[0])
print('vina:', vina.__version__)
print('py3Dmol:', getattr(py3Dmol, '__version__', 'n/a'))
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('matplotlib:', matplotlib.__version__)


## 2. Locate The Lab Folder And Prepared Input Files

This notebook assumes the receptor and ligand preparation has already been done and that the prepared files are available in `Docking_Vina/prepared`.


In [ ]:
from pathlib import Path

candidate_roots = [
    Path('/content/Lab_'),
    Path.cwd() / 'Lab_',
    Path.cwd(),
]

LAB_ROOT = None
for path in candidate_roots:
    if (path / 'Docking_Vina' / 'prepared').exists():
        LAB_ROOT = path
        break

if LAB_ROOT is None:
    raise FileNotFoundError('Could not locate the lab root containing Docking_Vina/prepared')

WORK_DIR = LAB_ROOT / 'Docking_Vina'
PREP_DIR = WORK_DIR / 'prepared'
RESULTS_DIR = WORK_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not PREP_DIR.exists():
    raise FileNotFoundError(f'Prepared folder not found: {PREP_DIR}')

LAB_ROOT, PREP_DIR, RESULTS_DIR


In [ ]:
prepared_inventory = pd.DataFrame([
    {
        'file': path.name,
        'suffix': path.suffix,
        'size_kb': round(path.stat().st_size / 1024, 1),
    }
    for path in sorted(PREP_DIR.iterdir())
    if path.is_file()
]).sort_values(['suffix', 'file']).reset_index(drop=True)

display(prepared_inventory)


## 3. Define The Receptors, Ligand, And Vina Box Inputs

Following the Vina tutorial, we define:
- the receptor `PDBQT` files
- the ligand `PDBQT` file
- the receptor `PDB` files used only for visualization and center-of-mass calculations
- the Vina box size and other run settings


In [ ]:
import subprocess
import shlex
import py3Dmol
from IPython.display import display

HOST_ORDER = ['BCD', 'HPBCD', 'SBEBCD']
VINA_BOX_SIZE_A = np.array([20.0, 20.0, 20.0], dtype=float)
EXHAUSTIVENESS = 32
N_POSES = 20
POSE_OVERLAY_COLORS = ['#1f77b4', '#4c78a8', '#72b7b2', '#54a24b', '#eeca3b', '#f58518', '#e45756', '#b279a2']

ATOMIC_MASSES = {
    'H': 1.008,
    'C': 12.011,
    'N': 14.007,
    'O': 15.999,
    'P': 30.974,
    'S': 32.060,
    'F': 18.998,
    'Cl': 35.450,
    'Br': 79.904,
    'I': 126.900,
    'Na': 22.990,
}

receptor_files = {host: PREP_DIR / f'{host}_rigid.pdbqt' for host in HOST_ORDER}
receptor_pdb_files = {host: PREP_DIR / f'{host}.pdb' for host in HOST_ORDER}
ligand_pdbqt = PREP_DIR / '57-DMF_flex.pdbqt'
ligand_pdb = PREP_DIR / '57-DMF_from_smiles.pdb'

for required_path in list(receptor_files.values()) + list(receptor_pdb_files.values()) + [ligand_pdbqt, ligand_pdb]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required prepared file: {required_path}')

print('Ligand PDBQT:', ligand_pdbqt)
for host in HOST_ORDER:
    print(host, receptor_files[host])


## 4. Write The Tutorial-Style Vina Config Files

The Vina basic-docking tutorial uses a text config file containing the box center and box size. Here we create one config file per host.


In [ ]:
host_boxes = {}
config_files = {}
box_rows = []

for host in HOST_ORDER:
    atom_rows = []
    for line in receptor_pdb_files[host].read_text().splitlines():
        if line.startswith(('ATOM', 'HETATM')):
            atom_name = line[12:16].strip()
            element = line[76:78].strip() or ''.join(ch for ch in atom_name if ch.isalpha())[:1].upper() or 'C'
            atom_rows.append((element, float(line[30:38]), float(line[38:46]), float(line[46:54])))

    host_df = pd.DataFrame(atom_rows, columns=['element', 'x', 'y', 'z'])
    coords = host_df[['x', 'y', 'z']].to_numpy(dtype=float)
    masses = np.array([ATOMIC_MASSES.get(el, 12.0) for el in host_df['element']], dtype=float)
    center = np.average(coords, axis=0, weights=masses).astype(float)
    size = VINA_BOX_SIZE_A.copy()

    host_boxes[host] = (center, size)
    config_path = RESULTS_DIR / f'{host}_vina_box.txt'
    config_path.write_text(
        f'center_x = {center[0]:.3f}\n'
        f'center_y = {center[1]:.3f}\n'
        f'center_z = {center[2]:.3f}\n'
        f'size_x = {size[0]:.1f}\n'
        f'size_y = {size[1]:.1f}\n'
        f'size_z = {size[2]:.1f}\n'
    )
    config_files[host] = config_path
    box_rows.append({
        'host': host,
        'center_x': center[0],
        'center_y': center[1],
        'center_z': center[2],
        'size_x': size[0],
        'size_y': size[1],
        'size_z': size[2],
        'config_file': str(config_path),
    })

box_table = pd.DataFrame(box_rows).round(3)
display(box_table)


## 5. Inspect The Pre-Docking Setup

Each view below shows the prepared receptor, the prepared ligand centered in the search box for visualization, and the tutorial-style Vina box.


In [ ]:
for host in HOST_ORDER:
    center, size = host_boxes[host]

    ligand_atom_lines = []
    for line in ligand_pdb.read_text().splitlines():
        if line.startswith(('ATOM', 'HETATM')):
            atom_name = line[12:16].strip()
            element = line[76:78].strip() or ''.join(ch for ch in atom_name if ch.isalpha())[:1].upper() or 'C'
            ligand_atom_lines.append((element, float(line[30:38]), float(line[38:46]), float(line[46:54]), line))

    ligand_df = pd.DataFrame([(row[0], row[1], row[2], row[3]) for row in ligand_atom_lines], columns=['element', 'x', 'y', 'z'])
    ligand_coords = ligand_df[['x', 'y', 'z']].to_numpy(dtype=float)
    ligand_masses = np.array([ATOMIC_MASSES.get(el, 12.0) for el in ligand_df['element']], dtype=float)
    ligand_com = np.average(ligand_coords, axis=0, weights=ligand_masses)
    shift = center - ligand_com

    shifted_ligand_lines = []
    for _, _, _, _, line in ligand_atom_lines:
        x = float(line[30:38]) + shift[0]
        y = float(line[38:46]) + shift[1]
        z = float(line[46:54]) + shift[2]
        shifted_ligand_lines.append(f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}")
    shifted_ligand_block = '\n'.join(shifted_ligand_lines) + '\n'

    half = size / 2.0
    corners = [
        center + np.array([-half[0], -half[1], -half[2]]),
        center + np.array([-half[0], -half[1],  half[2]]),
        center + np.array([-half[0],  half[1], -half[2]]),
        center + np.array([-half[0],  half[1],  half[2]]),
        center + np.array([ half[0], -half[1], -half[2]]),
        center + np.array([ half[0], -half[1],  half[2]]),
        center + np.array([ half[0],  half[1], -half[2]]),
        center + np.array([ half[0],  half[1],  half[2]]),
    ]
    edges = [
        (1, 2), (1, 3), (1, 5),
        (2, 4), (2, 6),
        (3, 4), (3, 7),
        (5, 6), (5, 7),
        (4, 8), (6, 8), (7, 8),
    ]
    box_block = '\n'.join(
        ['VINA_BOX', 'Codex', '', f"{8:>3}{12:>3}  0  0  0  0            999 V2000"]
        + [f"{coord[0]:10.4f}{coord[1]:10.4f}{coord[2]:10.4f} C   0  0  0  0  0  0  0  0  0  0  0  0" for coord in corners]
        + [f"{a:>3}{b:>3}{1:>3}  0  0  0  0" for a, b in edges]
        + ['M  END', '$$$$']
    ) + '\n'

    view = py3Dmol.view(width=700, height=430)
    view.setBackgroundColor('white')
    view.addModel(box_block, 'mol')
    view.setStyle({'model': 0}, {'stick': {'radius': 0.10, 'color': '#d62728'}})
    view.addModel(receptor_pdb_files[host].read_text(), 'pdb')
    view.setStyle({'model': 1}, {'stick': {'radius': 0.16, 'colorscheme': 'grayCarbon'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.10, 'color': 'lightgray'}, {'model': 1})
    view.addModel(shifted_ligand_block, 'pdb')
    view.setStyle({'model': 2}, {'stick': {'radius': 0.22, 'colorscheme': 'greenCarbon'}, 'sphere': {'scale': 0.18, 'colorscheme': 'greenCarbon'}})
    view.addLabel(
        f'{host} pre-docking setup\nPrepared receptor + ligand + Vina box',
        {'fontSize': 14, 'backgroundColor': 'white', 'fontColor': '#2c7c31', 'showBackground': True, 'inFront': True},
    )
    view.zoomTo()
    view.zoom(0.72)

    print()
    print(f'Pre-docking setup for {host}')
    display(view)


## 6. Run AutoDock Vina Using The Tutorial-Style Command-Line Workflow

In the spirit of the tutorial, the three docking runs are written out explicitly below, one receptor at a time, instead of being hidden inside a loop.


### 6.1 Dock `57-DMF` Into `BCD`

This follows the tutorial pattern directly:

```bash
vina --receptor BCD_rigid.pdbqt --ligand 57-DMF_flex.pdbqt \
     --config BCD_vina_box.txt --exhaustiveness 32 --out BCD_redocked_out.pdbqt
```


In [ ]:
host = 'BCD'
out_path = RESULTS_DIR / 'BCD_redocked_out.pdbqt'
log_path = RESULTS_DIR / 'BCD_redocked.log'

cmd = [
    'vina',
    '--receptor', str(receptor_files[host]),
    '--ligand', str(ligand_pdbqt),
    '--config', str(config_files[host]),
    '--exhaustiveness', str(EXHAUSTIVENESS),
    '--out', str(out_path),
    '--log', str(log_path),
    '--num_modes', str(N_POSES),
]

print('$', ' '.join(shlex.quote(str(arg)) for arg in cmd))
completed = subprocess.run(cmd, text=True, capture_output=True)
if completed.stdout.strip():
    print(completed.stdout[:4000])
if completed.stderr.strip():
    print(completed.stderr[:4000])
if completed.returncode != 0:
    raise RuntimeError(f'Vina failed for {host} with exit code {completed.returncode}')

rows = []
for line in out_path.read_text().splitlines():
    if line.startswith('REMARK VINA RESULT:'):
        parts = line.split()
        rows.append({
            'mode': len(rows) + 1,
            'affinity_kcal_mol': float(parts[3]),
            'rmsd_lb_A': float(parts[4]),
            'rmsd_ub_A': float(parts[5]),
        })
BCD_modes = pd.DataFrame(rows)
if BCD_modes.empty:
    raise RuntimeError('Vina returned no docked poses for BCD')

BCD_pose_models = []
current_rows = []
saw_model = False
for line in out_path.read_text().splitlines():
    if line.startswith('MODEL'):
        saw_model = True
        current_rows = []
        continue
    if line.startswith('ENDMDL'):
        if current_rows:
            BCD_pose_models.append(pd.DataFrame(current_rows, columns=['element', 'x', 'y', 'z']))
        current_rows = []
        continue
    if line.startswith(('ATOM', 'HETATM')):
        atom_name = line[12:16].strip()
        ad_type = line.split()[-1]
        element = 'C'
        if ad_type.startswith('Cl') or atom_name.startswith('Cl'):
            element = 'Cl'
        elif ad_type.startswith('Br') or atom_name.startswith('Br'):
            element = 'Br'
        else:
            letters = ''.join(ch for ch in atom_name if ch.isalpha())
            if letters:
                if len(letters) >= 2 and letters[:2].capitalize() in {'Cl', 'Br'}:
                    element = letters[:2].capitalize()
                else:
                    element = letters[0].upper()
            elif ad_type:
                element = ad_type[0].upper()
        current_rows.append((element, float(line[30:38]), float(line[38:46]), float(line[46:54])))
if current_rows:
    BCD_pose_models.append(pd.DataFrame(current_rows, columns=['element', 'x', 'y', 'z']))
if not saw_model and not BCD_pose_models:
    raise RuntimeError('Could not parse docked pose coordinates for BCD')

display(BCD_modes.round(3))


### 6.2 Dock `57-DMF` Into `HPBCD`


In [ ]:
host = 'HPBCD'
out_path = RESULTS_DIR / 'HPBCD_redocked_out.pdbqt'
log_path = RESULTS_DIR / 'HPBCD_redocked.log'

cmd = [
    'vina',
    '--receptor', str(receptor_files[host]),
    '--ligand', str(ligand_pdbqt),
    '--config', str(config_files[host]),
    '--exhaustiveness', str(EXHAUSTIVENESS),
    '--out', str(out_path),
    '--log', str(log_path),
    '--num_modes', str(N_POSES),
]

print('$', ' '.join(shlex.quote(str(arg)) for arg in cmd))
completed = subprocess.run(cmd, text=True, capture_output=True)
if completed.stdout.strip():
    print(completed.stdout[:4000])
if completed.stderr.strip():
    print(completed.stderr[:4000])
if completed.returncode != 0:
    raise RuntimeError(f'Vina failed for {host} with exit code {completed.returncode}')

rows = []
for line in out_path.read_text().splitlines():
    if line.startswith('REMARK VINA RESULT:'):
        parts = line.split()
        rows.append({
            'mode': len(rows) + 1,
            'affinity_kcal_mol': float(parts[3]),
            'rmsd_lb_A': float(parts[4]),
            'rmsd_ub_A': float(parts[5]),
        })
HPBCD_modes = pd.DataFrame(rows)
if HPBCD_modes.empty:
    raise RuntimeError('Vina returned no docked poses for HPBCD')

HPBCD_pose_models = []
current_rows = []
saw_model = False
for line in out_path.read_text().splitlines():
    if line.startswith('MODEL'):
        saw_model = True
        current_rows = []
        continue
    if line.startswith('ENDMDL'):
        if current_rows:
            HPBCD_pose_models.append(pd.DataFrame(current_rows, columns=['element', 'x', 'y', 'z']))
        current_rows = []
        continue
    if line.startswith(('ATOM', 'HETATM')):
        atom_name = line[12:16].strip()
        ad_type = line.split()[-1]
        element = 'C'
        if ad_type.startswith('Cl') or atom_name.startswith('Cl'):
            element = 'Cl'
        elif ad_type.startswith('Br') or atom_name.startswith('Br'):
            element = 'Br'
        else:
            letters = ''.join(ch for ch in atom_name if ch.isalpha())
            if letters:
                if len(letters) >= 2 and letters[:2].capitalize() in {'Cl', 'Br'}:
                    element = letters[:2].capitalize()
                else:
                    element = letters[0].upper()
            elif ad_type:
                element = ad_type[0].upper()
        current_rows.append((element, float(line[30:38]), float(line[38:46]), float(line[46:54])))
if current_rows:
    HPBCD_pose_models.append(pd.DataFrame(current_rows, columns=['element', 'x', 'y', 'z']))
if not saw_model and not HPBCD_pose_models:
    raise RuntimeError('Could not parse docked pose coordinates for HPBCD')

display(HPBCD_modes.round(3))


### 6.3 Dock `57-DMF` Into `SBEBCD`


In [ ]:
host = 'SBEBCD'
out_path = RESULTS_DIR / 'SBEBCD_redocked_out.pdbqt'
log_path = RESULTS_DIR / 'SBEBCD_redocked.log'

cmd = [
    'vina',
    '--receptor', str(receptor_files[host]),
    '--ligand', str(ligand_pdbqt),
    '--config', str(config_files[host]),
    '--exhaustiveness', str(EXHAUSTIVENESS),
    '--out', str(out_path),
    '--log', str(log_path),
    '--num_modes', str(N_POSES),
]

print('$', ' '.join(shlex.quote(str(arg)) for arg in cmd))
completed = subprocess.run(cmd, text=True, capture_output=True)
if completed.stdout.strip():
    print(completed.stdout[:4000])
if completed.stderr.strip():
    print(completed.stderr[:4000])
if completed.returncode != 0:
    raise RuntimeError(f'Vina failed for {host} with exit code {completed.returncode}')

rows = []
for line in out_path.read_text().splitlines():
    if line.startswith('REMARK VINA RESULT:'):
        parts = line.split()
        rows.append({
            'mode': len(rows) + 1,
            'affinity_kcal_mol': float(parts[3]),
            'rmsd_lb_A': float(parts[4]),
            'rmsd_ub_A': float(parts[5]),
        })
SBEBCD_modes = pd.DataFrame(rows)
if SBEBCD_modes.empty:
    raise RuntimeError('Vina returned no docked poses for SBEBCD')

SBEBCD_pose_models = []
current_rows = []
saw_model = False
for line in out_path.read_text().splitlines():
    if line.startswith('MODEL'):
        saw_model = True
        current_rows = []
        continue
    if line.startswith('ENDMDL'):
        if current_rows:
            SBEBCD_pose_models.append(pd.DataFrame(current_rows, columns=['element', 'x', 'y', 'z']))
        current_rows = []
        continue
    if line.startswith(('ATOM', 'HETATM')):
        atom_name = line[12:16].strip()
        ad_type = line.split()[-1]
        element = 'C'
        if ad_type.startswith('Cl') or atom_name.startswith('Cl'):
            element = 'Cl'
        elif ad_type.startswith('Br') or atom_name.startswith('Br'):
            element = 'Br'
        else:
            letters = ''.join(ch for ch in atom_name if ch.isalpha())
            if letters:
                if len(letters) >= 2 and letters[:2].capitalize() in {'Cl', 'Br'}:
                    element = letters[:2].capitalize()
                else:
                    element = letters[0].upper()
            elif ad_type:
                element = ad_type[0].upper()
        current_rows.append((element, float(line[30:38]), float(line[38:46]), float(line[46:54])))
if current_rows:
    SBEBCD_pose_models.append(pd.DataFrame(current_rows, columns=['element', 'x', 'y', 'z']))
if not saw_model and not SBEBCD_pose_models:
    raise RuntimeError('Could not parse docked pose coordinates for SBEBCD')

display(SBEBCD_modes.round(3))


### 6.4 Collect The Three Docking Runs Into A Single Summary Table


In [ ]:
redocking_outputs = {
    'BCD': {
        'host': 'BCD',
        'receptor_pdb': receptor_pdb_files['BCD'],
        'receptor_pdbqt': receptor_files['BCD'],
        'config_file': config_files['BCD'],
        'docked_pdbqt': RESULTS_DIR / 'BCD_redocked_out.pdbqt',
        'log_file': RESULTS_DIR / 'BCD_redocked.log',
        'modes': BCD_modes,
        'pose_models': BCD_pose_models,
    },
    'HPBCD': {
        'host': 'HPBCD',
        'receptor_pdb': receptor_pdb_files['HPBCD'],
        'receptor_pdbqt': receptor_files['HPBCD'],
        'config_file': config_files['HPBCD'],
        'docked_pdbqt': RESULTS_DIR / 'HPBCD_redocked_out.pdbqt',
        'log_file': RESULTS_DIR / 'HPBCD_redocked.log',
        'modes': HPBCD_modes,
        'pose_models': HPBCD_pose_models,
    },
    'SBEBCD': {
        'host': 'SBEBCD',
        'receptor_pdb': receptor_pdb_files['SBEBCD'],
        'receptor_pdbqt': receptor_files['SBEBCD'],
        'config_file': config_files['SBEBCD'],
        'docked_pdbqt': RESULTS_DIR / 'SBEBCD_redocked_out.pdbqt',
        'log_file': RESULTS_DIR / 'SBEBCD_redocked.log',
        'modes': SBEBCD_modes,
        'pose_models': SBEBCD_pose_models,
    },
}

redocking_scores = pd.DataFrame([
    {
        'host': 'BCD',
        'best_docked_affinity_kcal_mol': float(BCD_modes.iloc[0]['affinity_kcal_mol']),
        'best_mode_rmsd_lb_A': float(BCD_modes.iloc[0]['rmsd_lb_A']),
        'best_mode_rmsd_ub_A': float(BCD_modes.iloc[0]['rmsd_ub_A']),
        'n_reported_modes': len(BCD_modes),
    },
    {
        'host': 'HPBCD',
        'best_docked_affinity_kcal_mol': float(HPBCD_modes.iloc[0]['affinity_kcal_mol']),
        'best_mode_rmsd_lb_A': float(HPBCD_modes.iloc[0]['rmsd_lb_A']),
        'best_mode_rmsd_ub_A': float(HPBCD_modes.iloc[0]['rmsd_ub_A']),
        'n_reported_modes': len(HPBCD_modes),
    },
    {
        'host': 'SBEBCD',
        'best_docked_affinity_kcal_mol': float(SBEBCD_modes.iloc[0]['affinity_kcal_mol']),
        'best_mode_rmsd_lb_A': float(SBEBCD_modes.iloc[0]['rmsd_lb_A']),
        'best_mode_rmsd_ub_A': float(SBEBCD_modes.iloc[0]['rmsd_ub_A']),
        'n_reported_modes': len(SBEBCD_modes),
    },
]).sort_values('host').reset_index(drop=True)

display(redocking_scores.round(3))


## 7. Inspect The Pose-Level Vina Output

The columns below match the Vina basic-docking output table: affinity, lower-bound RMSD, and upper-bound RMSD relative to the best-ranked mode.


In [ ]:
for host in HOST_ORDER:
    print()
    print(f'Pose table for {host}')
    display(redocking_outputs[host]['modes'].round(3))


## 8. Visualize All Docked Poses For Each Host

Each host is shown with all reported docked ligand poses overlaid in a single interactive 3D view.


In [ ]:
for host in HOST_ORDER:
    pose_models = redocking_outputs[host]['pose_models']
    best_energy = redocking_outputs[host]['modes'].iloc[0]['affinity_kcal_mol']

    view = py3Dmol.view(width=700, height=430)
    view.addModel(receptor_pdb_files[host].read_text(), 'pdb')
    view.setStyle({'model': 0}, {'stick': {'radius': 0.16, 'colorscheme': 'grayCarbon'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.10, 'color': 'lightgray'}, {'model': 0})

    for pose_idx, pose_df in enumerate(pose_models, start=1):
        body = '\n'.join(
            f"{row.element:<2} {row.x: .6f} {row.y: .6f} {row.z: .6f}"
            for row in pose_df[['element', 'x', 'y', 'z']].itertuples(index=False)
        )
        xyz_block = f"{len(pose_df)}\n{host} mode {pose_idx}\n{body}\n"
        color = POSE_OVERLAY_COLORS[(pose_idx - 1) % len(POSE_OVERLAY_COLORS)]
        view.addModel(xyz_block, 'xyz')
        view.setStyle({'model': pose_idx}, {'stick': {'radius': 0.14, 'color': color}})

    view.addLabel(
        f'{host} all docked poses\n{len(pose_models)} modes\nBest {best_energy:.2f} kcal/mol',
        {'fontSize': 14, 'backgroundColor': 'white', 'fontColor': '#1f77b4', 'showBackground': True},
    )
    view.zoomTo()

    print()
    print(f'All docked poses for {host}')
    display(view)


## 9. Visualize The Best Docked Pose For Each Host


In [ ]:
for host in HOST_ORDER:
    best_pose_df = redocking_outputs[host]['pose_models'][0]
    best_energy = redocking_outputs[host]['modes'].iloc[0]['affinity_kcal_mol']

    body = '\n'.join(
        f"{row.element:<2} {row.x: .6f} {row.y: .6f} {row.z: .6f}"
        for row in best_pose_df[['element', 'x', 'y', 'z']].itertuples(index=False)
    )
    xyz_block = f"{len(best_pose_df)}\n{host} best mode\n{body}\n"

    view = py3Dmol.view(width=700, height=430)
    view.addModel(receptor_pdb_files[host].read_text(), 'pdb')
    view.setStyle({'model': 0}, {'stick': {'radius': 0.16, 'colorscheme': 'grayCarbon'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.10, 'color': 'lightgray'}, {'model': 0})
    view.addModel(xyz_block, 'xyz')
    view.setStyle({'model': 1}, {'stick': {'radius': 0.24, 'color': '#1f77b4'}, 'sphere': {'scale': 0.20, 'color': '#1f77b4'}})
    view.addLabel(
        f'{host} best pose\nAffinity {best_energy:.2f} kcal/mol',
        {'fontSize': 14, 'backgroundColor': 'white', 'fontColor': '#1f77b4', 'showBackground': True},
    )
    view.zoomTo()

    print()
    print(f'Best docked pose for {host}')
    display(view)


## 10. Summarize The Best Docking Affinities


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_df = redocking_scores.sort_values('best_docked_affinity_kcal_mol')
ax.bar(plot_df['host'], plot_df['best_docked_affinity_kcal_mol'], color=['#4c78a8', '#72b7b2', '#f58518'])
ax.set_title('Best Vina affinity by host')
ax.set_ylabel('kcal/mol')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


## 11. Where The Input And Output Files Live

This V02 notebook reads prepared inputs from:

- `Docking_Vina/prepared/`

and writes docking outputs to:

- `Docking_Vina/results/`

Tutorial-style files used or written here:
- `*_rigid.pdbqt`: receptor input for Vina
- `57-DMF_flex.pdbqt`: ligand input for Vina
- `*_vina_box.txt`: Vina config file with box center and size
- `*_redocked_out.pdbqt`: docked output poses from Vina
- `*_redocked.log`: Vina text log
